# Model Training - Salary Prediction Model

This notebook trains a Random Forest Regression model to predict the expected salary range in Lakhs Per Annum (LPA) for computer engineering job openings in India based on role, location, experience, and key skills.

In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

### Load and Clean Data for ML

In [2]:
df = pd.read_csv('../data/processed/jobs_cleaned.csv')
df_ml = df[df['avg_salary'] > 0].copy()
df_ml['target_lpa'] = df_ml['avg_salary'] / 100000.0
print(f"Total records with disclosed salaries for ML: {len(df_ml)}")

### Setup Preprocessing Pipeline

In [3]:
# Fill NaNs
df_ml['title'] = df_ml['title'].fillna('Software Engineer')
df_ml['location'] = df_ml['location'].fillna('Bengaluru')
df_ml['tagsAndSkills'] = df_ml['tagsAndSkills'].fillna('')
df_ml['minimumExperience'] = df_ml['minimumExperience'].fillna(0.0)
df_ml['maximumExperience'] = df_ml['maximumExperience'].fillna(df_ml['minimumExperience'] + 2.0)

X = df_ml[['title', 'location', 'minimumExperience', 'maximumExperience', 'tagsAndSkills']]
y = df_ml['target_lpa']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['minimumExperience', 'maximumExperience']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['title', 'location']),
        ('text', TfidfVectorizer(max_features=50, token_pattern=r'(?u)\b\w+\b'), 'tagsAndSkills')
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42))
])

### Train Test Split & Model Fitting

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)
print("Model training completed.")

### Model Evaluation

In [5]:
y_pred = model_pipeline.predict(X_test)
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f} LPA")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

### Save the Trained Model Pipeline

In [6]:
with open('../models/salary_model.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)
print("Model pipeline saved successfully to ../models/salary_model.pkl")